# Machine Learning para diagnóstico probabilistico de doenças degenerativas

## Introdução

A doença de Parkinson é uma doença degenerativa, e lentamente progressiva, de áreas específicas do cérebro. É caracterizada pelo tremor quando os músculos estão em repouso (tremor de repouso), aumento no tônus muscular (rigidez), lentidão dos movimentos voluntários e dificuldade de manter o equilíbrio (instabilidade postural). Em muitas pessoas, o pensamento torna-se comprometido ou desenvolve-se demência. A doença de Parkinson é a segunda doença degenerativa mais comum do sistema nervoso central após a doença de Alzheimer. 

Ela afeta:

- Cerca de 1 em 250 pessoas com 40 anos de idade ou mais
- Cerca de 1 em 100 pessoas com 65 anos de idade ou mais
- Cerca de 1 em 10 pessoas com 80 anos de idade ou mais

A doença de Parkinson normalmente começa entre os 50 e 79 anos. Raramente, ela ocorre em crianças ou adolescentes.

O objetivo é tentar prever a ocorrência da doença a partir de registros da voz de pacientes. Usaremos o diagnóstico probabilístico da doença de Parkinson com dados reais disponíveis publicamente e o modelo de Machine Learning será criado com o algoritmo Naive Bayes.

### Dados

Dataset: https://archive.ics.uci.edu/dataset/301/parkinson+speech+dataset+with+multiple+types+of+Audio+recordings

O conjunto de dados contém 46 atributos e 240 gravações de 80 participantes. Cada participante possui 3 gravações replicadas.Cada linha do dataset não pode ser usada independentemente, porque é uma das três repetições de um indivíduo.

A natureza dos dados é dependente de cada assunto, mas independente de um assunto para outro. Portanto, a técnica tradicional de aprendizado de máquina não pode ser aplicada a este conjunto de dados, pois essas técnicas são baseadas na natureza independente das instâncias. Existem 240 instâncias, mas para apenas 80 assuntos, portanto, eles não são independentes. 

Uma vez que as características são extraídas de múltiplas gravações de voz consecutivas do mesmo sujeito, em princípio, as características devem ser idênticas. As imperfeições da tecnologia e a própria variabilidade biológica resultam em características replicadas não idênticas que são mais semelhantes umas às outras do que características de assuntos diferentes.

### Pacotes

In [3]:
# Imports
import math
import sklearn
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import minmax_scale
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics
import warnings
warnings.filterwarnings('ignore')

### Carrega dados

In [4]:
# Carrega o dataset
df = pd.read_csv('dataset.csv')

In [5]:
df.head()

,ID,Recording,Status,Gender,Jitter_rel,Jitter_abs,Jitter_RAP,Jitter_PPQ,Shim_loc,Shim_dB,...,Delta3,Delta4,Delta5,Delta6,Delta7,Delta8,Delta9,Delta10,Delta11,Delta12
0,CONT-01,1,0,1,0.25546,0.000015,0.001467,0.001673,0.030256,0.26313,...,1.407701,1.417218,1.380352,1.420670,1.451240,1.440295,1.403678,1.405495,1.416705,1.354610
1,CONT-01,2,0,1,0.36964,0.000022,0.001932,0.002245,0.023146,0.20217,...,1.331232,1.227338,1.213377,1.352739,1.354242,1.365692,1.322870,1.314549,1.318999,1.323508
2,CONT-01,3,0,1,0.23514,0.000013,0.001353,0.001546,0.019338,0.16710,...,1.412304,1.324674,1.276088,1.429634,1.455996,1.368882,1.438053,1.388910,1.305469,1.305402
3,CONT-02,1,0,0,0.29320,0.000017,0.001105,0.001444,0.024716,0.20892,...,1.501200,1.534170,1.323993,1.496442,1.472926,1.643177,1.551286,1.638346,1.604008,1.621456
4,CONT-02,2,0,0,0.23075,0.000015,0.001073,0.001404,0.013119,0.11607,...,1.508468,1.334511,1.610694,1.685021,1.417614,1.574895,1.640088,1.533666,1.297536,1.382023


In [7]:
df['Recording'].value_counts()

Recording
1    80
2    80
3    80
Name: count, dtype: int64

In [8]:
df['Status'].value_counts()

Status
0    120
1    120
Name: count, dtype: int64

In [9]:
df.Gender.value_counts()

Gender
0    144
1     96
Name: count, dtype: int64

In [10]:
df.describe()

,Recording,Status,Gender,Jitter_rel,Jitter_abs,Jitter_RAP,Jitter_PPQ,Shim_loc,Shim_dB,Shim_APQ3,...,Delta3,Delta4,Delta5,Delta6,Delta7,Delta8,Delta9,Delta10,Delta11,Delta12
count,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,...,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000
mean,2.000000,0.500000,0.400000,0.583987,0.000044,0.003172,0.003532,0.038428,0.336832,0.021499,...,1.343238,1.348914,1.337717,1.341786,1.340895,1.343978,1.341472,1.331433,1.346381,1.346144
std,0.818203,0.501045,0.490922,0.535769,0.000045,0.003373,0.004449,0.023213,0.205905,0.013787,...,0.198174,0.212008,0.204978,0.209407,0.213689,0.211364,0.204085,0.211297,0.221484,0.208819
min,1.000000,0.000000,0.000000,0.148010,0.000007,0.000678,0.001036,0.007444,0.064989,0.003344,...,0.766458,0.840133,0.741690,0.759689,0.764649,0.762798,0.811942,0.777012,0.643132,0.748411
25%,1.000000,0.000000,0.000000,0.298260,0.000019,0.001551,0.001867,0.024336,0.211785,0.012910,...,1.208624,1.221101,1.197868,1.194186,1.194138,1.191881,1.189515,1.192397,1.202525,1.206559
50%,2.000000,0.500000,0.000000,0.481455,0.000035,0.002337,0.002870,0.032960,0.287885,0.018571,...,1.351015,1.342237,1.337233,1.338437,1.344416,1.335517,1.349243,1.333345,1.347297,1.331212
75%,3.000000,1.000000,1.000000,0.681685,0.000056,0.003678,0.003991,0.045475,0.399860,0.025784,...,1.488878,1.470287,1.485597,1.496845,1.490389,1.495355,1.475330,1.472589,1.506674,1.475235
max,3.000000,1.000000,1.000000,6.838200,0.000550,0.043843,0.065199,0.192600,1.747600,0.113240,...,1.860588,2.038241,1.785984,1.988090,1.872799,1.920131,1.943331,1.949679,1.918392,1.930103


### Pré-processamento dos dados

Dados contendo a média dos 3 registros padronizados

In [4]:
dados_finais = pd.read_csv('dados_reorganizados.csv')
dados_finais = dados_finais.drop(columns = ["Unnamed: 0"])
dados_finais.head(5)

,Status,Gender,Jitter_rel,Jitter_abs,Jitter_RAP,Jitter_PPQ,Shim_loc,Shim_dB,Shim_APQ3,Shim_APQ5,...,Delta3,Delta4,Delta5,Delta6,Delta7,Delta8,Delta9,Delta10,Delta11,Delta12
0,0,1,0.167313,0.188417,0.200556,0.133088,0.357736,0.352159,0.398112,0.387301,...,0.665718,0.510483,0.628141,0.635890,0.690788,0.650952,0.613847,0.615713,0.675877,0.605082
1,0,0,0.107455,0.150592,0.090730,0.062494,0.217872,0.215143,0.248739,0.252244,...,0.754630,0.660136,0.747662,0.791970,0.710295,0.753832,0.780908,0.789895,0.780707,0.715060
2,0,1,0.113051,0.166572,0.166969,0.095198,0.276710,0.274003,0.305235,0.316930,...,0.844668,0.646489,0.781302,0.620814,0.762391,0.666213,0.655040,0.766339,0.883611,0.752251
3,0,1,0.530916,0.580286,0.553443,0.430681,0.379989,0.378006,0.416113,0.372117,...,0.768217,0.611319,0.826046,0.683721,0.813216,0.707039,0.800814,0.608143,0.844810,0.788565
4,0,0,0.359656,0.547033,0.343497,0.298568,0.480625,0.472485,0.494463,0.496986,...,0.673844,0.643307,0.902141,0.749905,0.923264,0.657936,0.675329,0.618058,0.721695,0.762546


In [5]:
# Extrai os valores do dataframe
df_values = dados_finais.values

# A variável target é a coluna de índice zero
target = df_values[:, 0].astype(int)

# Para os atributos descartamos as colunas de índice 0 e índice 1 (status e gender)
atributos = np.delete(df_values, 0, 1)

In [6]:
atributos.shape

(80, 45)

In [7]:
target.shape

(80,)

In [8]:
# Divisão
x_treino, x_teste, y_treino, y_teste = train_test_split(atributos, target, test_size = 0.20, random_state = 42)

## Modelagem

### Aplicação do algoritmo Naive Bayes
 
 https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html

In [9]:
modelo = GaussianNB()

In [10]:
# Treina o modelo e faz as previsões
previsoes = modelo.fit(x_treino, y_treino).predict(x_teste)

In [11]:
# Calcula as métricas
cm = metrics.confusion_matrix(y_teste, previsoes)
print("\nMatriz de Confusão:")
print(cm)
print("\nAcurácia:", metrics.accuracy_score(y_teste, previsoes))
print("Precisão: ", metrics.precision_score(y_teste, previsoes))
print("Recall: ", metrics.recall_score(y_teste, previsoes))


Matriz de Confusão:
[[10  1]
 [ 0  5]]

Acurácia: 0.9375
Precisão:  0.8333333333333334
Recall:  1.0


In [12]:
# Previsões de classe
previsoes

array([0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1])

In [13]:
# Previsões como probabilidade
previsoes_proba = modelo.predict_proba(x_teste)

In [24]:
# Extraindo previsões de probabilidade da classe 1 (positiva)
previsoes_proba[:,1]

array([9.13044259e-22, 4.56380554e-05, 6.21725543e-21, 3.12319146e-20,
       1.32722602e-10, 1.78634437e-06, 1.00000000e+00, 1.00000000e+00,
       6.04626121e-16, 1.07161963e-09, 1.00000000e+00, 2.14816561e-18,
       1.00000000e+00, 4.68952265e-22, 1.00000000e+00, 1.00000000e+00])

In [25]:
# Métricas da curva ROC
fpr, tpr, thresholds = metrics.roc_curve(y_teste, previsoes_proba[:,1], pos_label = 1, sample_weight = None)

In [26]:
print("ROC AUC Score: ", metrics.roc_auc_score(y_teste, 
                                               previsoes_proba[:,1]))

ROC AUC Score:  0.9545454545454545


In [ ]:
# Gráfico ROC
plt.figure(figsize = (4,4))
plt.plot(fpr, tpr, linewidth = 2.0, label = "Modelo NB")
plt.plot([0,1], [0,1], 'k--', linewidth = 2.0, label = "Random")
plt.xlabel('Taxa de Falso Positivo')
plt.ylabel('Taxa de Verdadeiro Positivo')
plt.xlim([0,1])
plt.ylim([0,1])
plt.title('ROC')
plt.legend(loc = "lower right")
plt.show();

##  Sensibilidade e Especificidade

In [31]:
# Import
from sklearn.metrics import confusion_matrix

In [36]:
def calcula_sensibilidade(y_verdadeiro, y_predito):
    
    # Gera a matriz de confusão
    tn, fp, fn, tp = confusion_matrix(y_verdadeiro, y_predito).ravel()

    # Calcula a sensibilidade
    sensibilidade = tp / (tp + fn)

    return sensibilidade

def calcula_especificidade(y_verdadeiro, y_predito):
    
    # Gera a matriz de confusão
    tn, fp, fn, tp = confusion_matrix(y_verdadeiro, y_predito).ravel()

    # Calcula a especificidade
    especificidade = tn / (tn + fp)

    return especificidade

In [38]:
calcula_sensibilidade(y_teste, previsoes)

np.float64(1.0)

In [39]:
calcula_especificidade(y_teste, previsoes)

np.float64(0.9090909090909091)